# 05 Transformer 与大语言模型 · 代码示例

**章节**: 05_transformer_and_llm.md
**学习目标**: 亲手跑通一个简化版 Self-Attention 权重计算
**A3 关联**: A3 系统的多智能体「大脑」基于 Transformer 架构

运行本 notebook 需要 numpy。在 A3 项目中,我们用同样的原理做 Embedding、向量检索。

In [ ]:
import numpy as np

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / e_x.sum(axis=-1, keepdims=True)

def simplified_self_attention(X, W_q, W_k, W_v):
    Q = np.dot(X, W_q)
    K = np.dot(X, W_k)
    V = np.dot(X, W_v)
    d_k = Q.shape[-1]
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    weights = softmax(scores)
    output = np.dot(weights, V)
    return weights, output

### 解读

- `Q/K/V` 三个矩阵由输入 `X` 与权重矩阵相乘得到,代表查询、键、值
- `QK^T / sqrt(d_k)` 是注意力得分的核心公式,除以根号 d_k 防止数值爆炸
- `softmax` 把分数归一化为概率分布,每行加起来等于 1
- 最终输出 = 注意力权重 × V,即对每个位置的 V 加权求和

In [ ]:
np.random.seed(42)
X = np.random.randn(4, 8)        # 4 个词,每个 8 维
W_q = np.random.randn(8, 4)
W_k = np.random.randn(8, 4)
W_v = np.random.randn(8, 4)

weights, out = simplified_self_attention(X, W_q, W_k, W_v)
print("注意力权重矩阵 (4x4):")
print(np.round(weights, 2))
print("\n每行和:", weights.sum(axis=1).round(2))

### 拓展练习

1. 试着把 `d_model` 改为 16,观察权重矩阵的分布是否更平滑
2. 实现 Multi-Head 版本:把 `d_model` 切成 4 个头,每个头独立计算注意力后拼接
3. 在 A3 系统中,这个简化版注意力可以用作 RAG 检索的「轻量级 Embedding」
